# /whatifworks-longform — $0 Google Colab T4 Batch Chatterbox TTS

Generates 128kbps `ch01.mp3`–`ch05.mp3` using `whatif_main_v1` (`voice_ref.wav`) and dialed-back long-form `MOOD_PRESETS` (`0.68 / 0.55 / 0.38`).

In [ ]:
# Cell 1: Install dependencies & load colab_tts_bundle.zip
!pip install -U chatterbox-tts soundfile numpy
!pip uninstall -y torchvision
!which ffmpeg || (apt-get -qq update && apt-get -qq install -y ffmpeg)

import os, shutil, zipfile
from pathlib import Path
import torch

assert torch.cuda.is_available(), 'Enable GPU runtime: Runtime -> Change runtime type -> T4 GPU'
print('Using GPU:', torch.cuda.get_device_name(0))

WORK_DIR = Path('/content/tts_work')
if WORK_DIR.exists():
    shutil.rmtree(WORK_DIR)
WORK_DIR.mkdir(parents=True, exist_ok=True)

drive_bundle = Path('/content/drive/MyDrive/colab_tts_bundle.zip')
local_bundle = Path('/content/colab_tts_bundle.zip')

if not drive_bundle.is_file():
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
    except Exception as e:
        print('Drive mount skipped:', e)

if drive_bundle.is_file():
    bundle_path = drive_bundle
    print(f'Found bundle in Google Drive: {bundle_path}')
elif local_bundle.is_file():
    bundle_path = local_bundle
    print(f'Found local bundle: {bundle_path}')
else:
    from google.colab import files
    print('Please upload colab_tts_bundle.zip (generated by scripts/pack_colab_tts.py)...')
    uploaded = files.upload()
    fname = next(iter(uploaded.keys()))
    bundle_path = Path('/content') / fname

with zipfile.ZipFile(bundle_path, 'r') as zf:
    zf.extractall(WORK_DIR)

print('Extracted files:', sorted(str(p.relative_to(WORK_DIR)) for p in WORK_DIR.rglob('*') if p.is_file()))

In [ ]:
# Cell 2: Run ChatterboxTTS on CUDA with pronunciation sanitizer & dialed-back MOOD_PRESETS
import json, re, time
from pathlib import Path
import numpy as np
import torch
from chatterbox.tts import ChatterboxTTS

WORK_DIR = Path('/content/tts_work')

MOOD_PRESETS = {
    'hook':    {'exaggeration': 0.68, 'cfg_weight': 0.38, 'temperature': 0.75},
    'panic':   {'exaggeration': 0.68, 'cfg_weight': 0.38, 'temperature': 0.75},
    'explain': {'exaggeration': 0.55, 'cfg_weight': 0.45, 'temperature': 0.72},
    'deadpan': {'exaggeration': 0.38, 'cfg_weight': 0.52, 'temperature': 0.68},
}

PRONUNCIATION_MAP = [
    (r"\bWatt's\b", "Wott's"),
    (r"\bWatts'\b", "Wotts'"),
    (r"\bWatts\b", "Wotts"),
    (r"\bWatt\b", "Wott"),
    (r"\bwatt's\b", "wott's"),
    (r"\bwatts\b", "wotts"),
    (r"\bwatt\b", "wott"),
]

_ONES = [
    'zero', 'one', 'two', 'three', 'four', 'five', 'six', 'seven', 'eight', 'nine',
    'ten', 'eleven', 'twelve', 'thirteen', 'fourteen', 'fifteen', 'sixteen',
    'seventeen', 'eighteen', 'nineteen',
]
_TENS = ['', '', 'twenty', 'thirty', 'forty', 'fifty', 'sixty', 'seventy', 'eighty', 'ninety']
_SCALES = [(10**12, 'trillion'), (10**9, 'billion'), (10**6, 'million'), (10**3, 'thousand')]

def _int_to_words(n):
    if n < 0: return 'minus ' + _int_to_words(-n)
    if n < 20: return _ONES[n]
    if n < 100:
        t, o = divmod(n, 10)
        return _TENS[t] + (f'-{_ONES[o]}' if o else '')
    if n < 1000:
        h, r = divmod(n, 100)
        return f'{_ONES[h]} hundred' + (f' {_int_to_words(r)}' if r else '')
    for sv, sn in _SCALES:
        if n >= sv:
            q, r = divmod(n, sv)
            return f'{_int_to_words(q)} {sn}' + (f' {_int_to_words(r)}' if r else '')
    return str(n)

def _number_to_words(num_str):
    clean = num_str.replace(',', '')
    if '.' in clean:
        ip, fp = clean.split('.', 1)
        iw = _int_to_words(int(ip)) if ip else 'zero'
        fw = ' '.join(_ONES[int(d)] for d in fp if d.isdigit())
        return f'{iw} point {fw}' if fw else iw
    return _int_to_words(int(clean))

def sanitize_pronunciation(text):
    text = re.sub(r'(?<![\w.])(\d[\d,]*(?:\.\d+)?)\s*%', lambda m: f"{_number_to_words(m.group(1))} percent", text)
    text = re.sub(r'\b(\d+\.\d+)\b', lambda m: _number_to_words(m.group(1)), text)
    for pat, rep in PRONUNCIATION_MAP:
        text = re.sub(pat, rep, text)
    return text.strip()

def split_beat(text, max_chars=280):
    text = re.sub(r'\s+', ' ', text).strip()
    if len(text) <= max_chars:
        return [text]
    parts, cur = [], ''
    for sent in re.split(r'(?<=[.!?])\s+', text):
        if not sent: continue
        if len(cur) + len(sent) + 1 <= max_chars:
            cur = f'{cur} {sent}'.strip()
        else:
            if cur: parts.append(cur)
            cur = sent
    if cur: parts.append(cur)
    return parts

def trim_silence(samples, sr, thresh=0.008, pad_ms=25):
    idx = np.where(np.abs(samples) > thresh)[0]
    if len(idx) == 0:
        return samples
    pad = int(sr * pad_ms / 1000)
    lo = max(0, idx[0] - pad)
    hi = min(len(samples), idx[-1] + pad)
    return samples[lo:hi]

t0 = time.time()
model = ChatterboxTTS.from_pretrained(device='cuda')
SR = model.sr
VOICE_REF = str(WORK_DIR / 'voice_ref.wav')
print(f'Loaded ChatterboxTTS on CUDA (sr={SR}) in {time.time() - t0:.1f}s')

CHAPTER_AUDIO = {}
for script_path in sorted((WORK_DIR / 'scripts').glob('ch*.json')):
    ch_id = script_path.stem
    beats = json.loads(script_path.read_text(encoding='utf-8'))
    print(f'\n=== Synthesizing {ch_id} ({len(beats)} beats) ===')
    beat_wavs = []
    for bi, item in enumerate(beats):
        mood = (item.get('mood', 'explain') if isinstance(item, dict) else 'explain').lower()
        raw_text = item.get('text', '') if isinstance(item, dict) else str(item)
        preset = MOOD_PRESETS.get(mood, MOOD_PRESETS['explain'])
        clean_text = sanitize_pronunciation(raw_text)
        for sub in split_beat(clean_text):
            print(f'  [{ch_id} b{bi+1}] mood={mood} exag={preset["exaggeration"]} cfg={preset["cfg_weight"]}: {sub[:65]}...')
            with torch.inference_mode():
                wav_t = model.generate(
                    sub,
                    audio_prompt_path=VOICE_REF,
                    exaggeration=preset['exaggeration'],
                    cfg_weight=preset['cfg_weight'],
                    temperature=preset['temperature'],
                )
            arr = wav_t.squeeze().detach().cpu().numpy().astype(np.float32)
            beat_wavs.append(trim_silence(arr, SR))
    CHAPTER_AUDIO[ch_id] = beat_wavs

print(f'\nAll chapters synthesized in {time.time() - t0:.1f}s!')

In [ ]:
# Cell 3: Stitch beats (0.12s gap + 0.35s end pad), encode 128kbps MP3s, zip & download
import subprocess, zipfile
import numpy as np
import soundfile as sf

BEAT_GAP_SEC = 0.12
END_PAD_SEC = 0.35
OUT_DIR = Path('/content/tts_out')
OUT_DIR.mkdir(parents=True, exist_ok=True)

gap_samples = np.zeros(int(BEAT_GAP_SEC * SR), dtype=np.float32)
pad_samples = np.zeros(int(END_PAD_SEC * SR), dtype=np.float32)

mp3_files = []
for ch_id, beat_wavs in sorted(CHAPTER_AUDIO.items()):
    parts = []
    for idx, arr in enumerate(beat_wavs):
        parts.append(arr)
        if idx < len(beat_wavs) - 1:
            parts.append(gap_samples)
    parts.append(pad_samples)
    full = np.concatenate(parts)

    # Normalize peak to -1.0 dBFS
    peak = float(np.max(np.abs(full))) if len(full) else 1.0
    if peak > 0:
        full = full * (0.89 / peak)

    wav_path = OUT_DIR / f'{ch_id}.wav'
    mp3_path = OUT_DIR / f'{ch_id}.mp3'
    sf.write(str(wav_path), full, SR)
    subprocess.run([
        'ffmpeg', '-y', '-v', 'error', '-i', str(wav_path),
        '-codec:a', 'libmp3lame', '-b:a', '128k', str(mp3_path)
    ], check=True)
    wav_path.unlink()
    dur = len(full) / SR
    mp3_files.append(mp3_path)
    print(f'[encoded] {mp3_path.name}: {dur:.2f}s ({mp3_path.stat().st_size / 1024:.1f} KB)')

out_zip = Path('/content/colab_audio_out.zip')
with zipfile.ZipFile(out_zip, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for mp3 in mp3_files:
        zf.write(mp3, arcname=mp3.name)

print(f'\nPackaged {len(mp3_files)} MP3s -> {out_zip}')
drive_mydrive = Path('/content/drive/MyDrive')
if drive_mydrive.is_dir():
    import shutil as _shutil
    drive_out = drive_mydrive / 'colab_audio_out.zip'
    _shutil.copy2(out_zip, drive_out)
    print(f'Saved to Google Drive: {drive_out}')
try:
    from google.colab import files
    files.download(str(out_zip))
except ImportError:
    print('Not in Colab; download colab_audio_out.zip manually.')